# 01 — Build the atlas

This notebook makes one count matrix from three sources and saves it as `atlas.h5ad`.

| source | cells | where the counts come from |
|---|---|---|
| Yoon et al. 2023 integrated data (Cattenoz, Cho Drop-seq, Fu, Girard, Leitão, Tattikota) | 125,402 | figshare 24014010 |
| 10x w1118 (Yoon et al. 2023) | 2,006 | GEO GSE184781; the cells listed in the authors' metadata (figshare 16585805) |
| AML1-ETO / RasV12 / control (this study) | 30,000 | GEO GSE325999 |

Steps:
1. Read the FlyBase gene tables.
2. Read the Yoon data and give every gene its FlyBase gene ID.
3. Give every Yoon cell its sample and its published cell-type label.
4. Read the w1118 and AML cells from GEO.
5. Join everything, remove low-quality and unlabelled cells, normalise and save.

In [ ]:
import re

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scipy import sparse

from settings import ATLAS_FILE, INPUT_DIR, ROOT

## Input files and settings

In [ ]:
YOON_DIR = ROOT / "data" / "raw" / "yoon_integrated_figshare_24014010"
W1118_METADATA = ROOT / "data" / "raw" / "yoon_w1118_10x_figshare_16585805" / "metadata.txt"
W1118_DIR = ROOT / "data" / "raw" / "GSE184781"
AML_DIR = ROOT / "data" / "raw" / "GSE325999"
FLYBASE_DIR = ROOT / "data" / "reference" / "flybase_fb_2026_02"

SAMPLE_SHEET = INPUT_DIR / "samples.tsv"
PUBLISHED_LABELS = INPUT_DIR / "published_labels.tsv.gz"

INDROPS_DATASET = "tattikota_2020_indrops"      # Yoon's name for the Tattikota inDrops libraries
TRANSGENES = ["EGFP", "gal4"]                   # transgene features of the AML count matrix
MIN_GENES = 200                                 # cells with this many detected genes or fewer are removed

## 1. FlyBase gene tables

The Yoon and w1118 data name genes by symbol. Symbols change between FlyBase releases, so every gene gets its
FlyBase gene ID (release FB2026_02). Three look-up tables are built here:

- current symbol → gene ID (a symbol that belongs to more than one gene is left out);
- old symbol (synonym) → gene ID (again only synonyms that belong to exactly one gene);
- retired (secondary) gene ID → current (primary) gene ID.

In [ ]:
ANNOTATION_COLUMNS = [
    "symbol",
    "organism",
    "primary_fbgn",
    "secondary_fbgn",
    "annotation_id",
    "secondary_annotation_id",
]

annotation = pd.read_csv(
    FLYBASE_DIR / "fbgn_annotation_ID_fb_2026_02.tsv.gz",
    sep="\t",
    comment="#",
    header=None,
    dtype=str,
    keep_default_na=False,
    names=ANNOTATION_COLUMNS,
)
annotation = annotation[annotation["organism"].eq("Dmel")]

# current symbol -> gene ID
genes_per_symbol = annotation.groupby("symbol")["primary_fbgn"].nunique()
ambiguous_symbols = genes_per_symbol.index[genes_per_symbol > 1]

unambiguous = annotation[~annotation["symbol"].isin(ambiguous_symbols)]
symbol_to_gene = unambiguous.drop_duplicates("symbol").set_index("symbol")["primary_fbgn"]

# gene ID -> current symbol
gene_to_symbol = annotation.drop_duplicates("primary_fbgn").set_index("primary_fbgn")["symbol"]

# retired gene ID -> current gene ID (a gene can have several retired IDs, separated by commas)
secondary = annotation[["primary_fbgn", "secondary_fbgn"]].copy()
secondary["secondary_fbgn"] = secondary["secondary_fbgn"].str.split(",")
secondary = secondary.explode("secondary_fbgn")
secondary["secondary_fbgn"] = secondary["secondary_fbgn"].str.strip()
secondary = secondary[secondary["secondary_fbgn"].ne("")]

secondary_to_primary = secondary.drop_duplicates("secondary_fbgn").set_index("secondary_fbgn")["primary_fbgn"]

In [ ]:
SYNONYM_COLUMNS = [
    "primary_fbid",
    "organism",
    "current_symbol",
    "current_fullname",
    "fullname_synonyms",
    "symbol_synonyms",
]

# The synonym table has two malformed lines; they are skipped.
synonyms = pd.read_csv(
    FLYBASE_DIR / "fb_synonym_fb_2026_02.tsv.gz",
    sep="\t",
    comment="#",
    header=None,
    na_filter=False,
    names=SYNONYM_COLUMNS,
    on_bad_lines="skip",
    engine="python",
)
synonyms = synonyms[synonyms["organism"].eq("Dmel") & synonyms["primary_fbid"].str.startswith("FBgn")]

# one row per (synonym, gene); a gene's synonyms are separated by "|"
synonym_pairs = synonyms[["primary_fbid", "symbol_synonyms"]].copy()
synonym_pairs["synonym"] = synonym_pairs["symbol_synonyms"].str.split("|")
synonym_pairs = synonym_pairs.explode("synonym")
synonym_pairs["synonym"] = synonym_pairs["synonym"].str.strip()
synonym_pairs = synonym_pairs[synonym_pairs["synonym"].ne("")]

genes_per_synonym = synonym_pairs.groupby("synonym")["primary_fbid"].nunique()
ambiguous_synonyms = genes_per_synonym.index[genes_per_synonym > 1]

unambiguous = synonym_pairs[~synonym_pairs["synonym"].isin(ambiguous_synonyms)]
synonym_to_gene = unambiguous.drop_duplicates("synonym").set_index("synonym")["primary_fbid"]

print(f"{len(symbol_to_gene):,} current symbols")
print(f"{len(synonym_to_gene):,} synonyms")
print(f"{len(secondary_to_primary):,} retired gene IDs")

Two helper functions, used for every source:

- `to_gene_ids` looks each symbol up as a current symbol first, then as a synonym, and replaces a retired gene ID by
  the current one. Symbols that FlyBase does not know stay in the atlas under the name `UNMAPPED:<symbol>`.
- `sum_same_genes` adds up the counts of columns that end up with the same gene ID, so every gene ID is one column.
  It builds a table with one row per old column and one column per gene ID; multiplying the counts by it does the sum.

In [ ]:
def to_gene_ids(symbols):
    """FlyBase gene ID of each symbol (UNMAPPED:<symbol> when FlyBase does not know it)."""
    symbols = pd.Series(symbols, index=symbols)

    gene_ids = symbols.map(symbol_to_gene)
    not_current = gene_ids.isna()
    gene_ids[not_current] = symbols[not_current].map(synonym_to_gene)

    is_mapped = gene_ids.notna()
    gene_ids = gene_ids.map(lambda gene: secondary_to_primary.get(gene, gene))
    gene_ids = gene_ids.where(is_mapped, "UNMAPPED:" + symbols)

    return gene_ids.to_numpy()


def sum_same_genes(counts, gene_ids):
    """Add up the columns of `counts` that have the same gene ID; returns the new counts and their gene IDs."""
    unique_gene_ids, column_of_gene = np.unique(gene_ids, return_inverse=True)

    summing = sparse.csr_matrix(
        (np.ones(len(gene_ids), dtype=np.float32), (np.arange(len(gene_ids)), column_of_gene)),
        shape=(len(gene_ids), len(unique_gene_ids)),
    )

    return sparse.csr_matrix(counts @ summing), unique_gene_ids

## 2. Yoon data: counts and gene IDs

The matrix file has genes in rows and cells in columns, so it is turned (`.T`) to cells × genes.
`keep_default_na=False` keeps the gene that is really called "nan" (*nanchung*) as text.

In [ ]:
yoon_metadata = pd.read_csv(YOON_DIR / "metadata.txt", sep="\t", low_memory=False).set_index("Barcode")

yoon_cells = pd.read_csv(YOON_DIR / "barcodes.tsv.gz", header=None, dtype=str, keep_default_na=False)
yoon_cells = yoon_cells[0].to_numpy()

yoon_symbols = pd.read_csv(YOON_DIR / "genes.tsv.gz", header=None, dtype=str, keep_default_na=False)
yoon_symbols = yoon_symbols[0].to_numpy()

yoon_counts = sparse.csr_matrix(sc.read_mtx(YOON_DIR / "matrix.mtx.gz", dtype="float32").T.X)

In [ ]:
yoon_gene_ids = to_gene_ids(yoon_symbols)
yoon_counts, yoon_gene_ids = sum_same_genes(yoon_counts, yoon_gene_ids)

n_unmapped = sum(gene.startswith("UNMAPPED:") for gene in yoon_gene_ids)
print(f"Yoon data: {yoon_counts.shape[0]:,} cells")
print(f"{len(yoon_symbols):,} gene symbols -> {yoon_counts.shape[1]:,} gene IDs")
print(f"{n_unmapped} symbols are not in FlyBase")

## 3. Yoon cells: sample, Yoon's cell type, published label

- **Sample:** the sample sheet (`samples.tsv`) says which of our samples each Yoon library is (`yoon_library`).
- **Yoon's cell type:** `celltype_big` (PM, LM, CC, PH, ...) and `subtype_big` (PM 1, LM 2, ...) of the Yoon metadata.
- **Published label:** the label the cell has in its own study (`published_labels.tsv.gz`). That table uses our cell
  names, the Yoon data use other names, so cells are matched by sample and by the barcode inside the cell name.

In [ ]:
samples = pd.read_csv(SAMPLE_SHEET, sep="\t")

library_to_sample = samples.dropna(subset=["yoon_library"]).set_index("yoon_library")["sample_id"]

yoon_obs = pd.DataFrame(index=yoon_cells)
yoon_obs["sample_id"] = yoon_metadata["Library"].map(library_to_sample)
yoon_obs["yoon_celltype"] = yoon_metadata["celltype_big"]
yoon_obs["yoon_subtype"] = yoon_metadata["subtype_big"]

In [ ]:
def cell_barcode(cell_name, indrops):
    """The barcode inside a cell name.

    10x and Drop-seq: the first run of 12 to 16 bases. inDrops: two halves of 8 bases, joined by "-".
    """
    if not indrops:
        return re.search(r"[ACGTN]{12,16}", cell_name).group(0)

    if "-indrops_" in cell_name:
        first, second = cell_name.split("-")[:2]
    else:
        first, second = re.findall(r"[ACGTN]{8}", cell_name)[-2:]

    return f"{first}-{second}"

In [ ]:
published = pd.read_csv(PUBLISHED_LABELS, sep="\t", dtype=str)

published["barcode"] = [
    cell_barcode(cell_name, indrops="indrop" in sample)
    for cell_name, sample in zip(published["cell_id"], published["sample_id"])
]

# a (sample, barcode) pair that occurs twice cannot be matched to one cell, so both rows are left out
matchable = published.drop_duplicates(["sample_id", "barcode"], keep=False)
label_of_barcode = matchable.set_index(["sample_id", "barcode"])["published_label"]

yoon_is_indrops = yoon_metadata["dataset"].reindex(yoon_cells).eq(INDROPS_DATASET)

yoon_obs["barcode"] = [
    cell_barcode(cell_name, indrops=indrops)
    for cell_name, indrops in zip(yoon_cells, yoon_is_indrops)
]

yoon_keys = pd.MultiIndex.from_arrays([yoon_obs["sample_id"], yoon_obs["barcode"]])
yoon_obs["published_label"] = label_of_barcode.reindex(yoon_keys).to_numpy()
yoon_obs = yoon_obs.drop(columns="barcode")

yoon = ad.AnnData(X=yoon_counts, obs=yoon_obs, var=pd.DataFrame(index=yoon_gene_ids))

## 4. The w1118 and AML cells (GEO)

**w1118 (GEO GSE184781):** the count matrix names genes by symbol, so they get their gene IDs as the Yoon data did.
Only the cells in the authors' own metadata are kept (2,006 of 2,016), with the authors' cell types; their sample
is the metadata's `Library` (normal-1, normal-2, normal-3).

In [ ]:
w1118_metadata = pd.read_csv(W1118_METADATA, sep="\t", low_memory=False).set_index("Barcode")

w1118_cells = pd.read_csv(W1118_DIR / "GSE184781_barcodes.tsv.gz", header=None, dtype=str)[0].to_numpy()

w1118_symbols = pd.read_csv(W1118_DIR / "GSE184781_genes.tsv.gz", header=None, dtype=str, keep_default_na=False)
w1118_symbols = w1118_symbols[0].to_numpy()

w1118_counts = sparse.csr_matrix(sc.read_mtx(W1118_DIR / "GSE184781_matrix.mtx.gz", dtype="float32").T.X)
w1118_counts, w1118_gene_ids = sum_same_genes(w1118_counts, to_gene_ids(w1118_symbols))

In [ ]:
in_metadata = np.isin(w1118_cells, w1118_metadata.index)
w1118_cells = w1118_cells[in_metadata]

label_of_cell = published.set_index("cell_id")["published_label"]

w1118_obs = pd.DataFrame(index=w1118_cells)
w1118_obs["sample_id"] = w1118_metadata.loc[w1118_cells, "Library"].to_numpy()
w1118_obs["yoon_celltype"] = w1118_metadata.loc[w1118_cells, "celltype_big"].to_numpy()
w1118_obs["yoon_subtype"] = w1118_metadata.loc[w1118_cells, "subtype_big"].to_numpy()
w1118_obs["published_label"] = label_of_cell.reindex(w1118_cells).to_numpy()

w1118 = ad.AnnData(X=w1118_counts[in_metadata], obs=w1118_obs, var=pd.DataFrame(index=w1118_gene_ids))

print(f"w1118: {w1118.n_obs:,} cells")

**AML (GEO GSE325999):** a text table of counts with genes (FlyBase gene IDs) in rows and cells in columns, and a
metadata table with one row per cell, in the same order: barcode, sample (AML1-ETO, RasV12, Empty_control) and
cluster (PM1–PM9, LM1–2, CC1–2). The cluster is the cell's published label. The same barcode occurs in more than
one sample, so a cell is named `<barcode>-<sample>`. The count table is large, so it is read in parts of 2,000
genes. Retired gene IDs are replaced by the current ones.

In [ ]:
aml_metadata = pd.read_csv(AML_DIR / "GSE325999_metadata_GEO.txt.gz", sep="\t")
aml_metadata = aml_metadata.dropna(subset=["barcode"])                  # the file ends with empty lines

aml_parts = []
aml_genes = []

for part in pd.read_csv(AML_DIR / "GSE325999_matrix_GEO.txt.gz", sep="\t", index_col=0, chunksize=2000):
    aml_genes.extend(part.index)
    aml_parts.append(sparse.csr_matrix(part.to_numpy(dtype=np.float32)))

aml_counts = sparse.csr_matrix(sparse.vstack(aml_parts).T)

In [ ]:
aml_gene_ids = np.array([secondary_to_primary.get(gene, gene) for gene in aml_genes])
aml_counts, aml_gene_ids = sum_same_genes(aml_counts, aml_gene_ids)

aml_cells = (aml_metadata["barcode"] + "-" + aml_metadata["sample"]).to_numpy()

aml_obs = pd.DataFrame(index=aml_cells)
aml_obs["sample_id"] = aml_metadata["sample"].to_numpy()
aml_obs["published_label"] = aml_metadata["cluster"].to_numpy()

aml = ad.AnnData(X=aml_counts, obs=aml_obs, var=pd.DataFrame(index=aml_gene_ids))

print(f"AML: {aml.n_obs:,} cells")

## 5. Join, filter, normalise, save

The three parts are joined on the union of their genes (a gene missing from one part counts as 0 there).
The sample sheet then adds each cell's dataset, experiment, condition, compartment, technology and its
Harmony batch (used in notebook 02).

In [ ]:
atlas = ad.concat({"yoon": yoon, "w1118": w1118, "aml": aml}, join="outer")
atlas.X = sparse.csr_matrix(atlas.X, dtype=np.float32)

# store every cell's entries in gene order, so later sums round the same way in every run
atlas.X.sort_indices()

sample_columns = ["dataset", "experiment", "condition", "compartment", "technology", "harmony_batch"]
atlas.obs = atlas.obs.join(samples.set_index("sample_id")[sample_columns], on="sample_id")

column_order = ["dataset", "sample_id", *sample_columns[1:], "published_label", "yoon_celltype", "yoon_subtype"]
atlas.obs = atlas.obs[column_order]

Two kinds of cells are removed:

- cells with **≤ 200 detected genes**;
- cells **without a published label**: cells that are in the Yoon data but that their own study did not annotate.

After that, genes without a single count are dropped.

In [ ]:
genes_per_cell = atlas.X.getnnz(axis=1)
few_genes = genes_per_cell <= MIN_GENES
unlabelled = atlas.obs["published_label"].isna().to_numpy()
removed = few_genes | unlabelled

removal = pd.DataFrame(
    {
        "cells": 1,
        f"<= {MIN_GENES} genes": few_genes,
        "no published label": unlabelled,
        "removed": removed,
    }
).astype(int)

removal = removal.groupby(atlas.obs["dataset"].to_numpy()).sum()
removal["kept"] = removal["cells"] - removal["removed"]
display(removal)

atlas = atlas[~removed].copy()

has_counts = np.asarray(atlas.X.sum(axis=0)).ravel() > 0
atlas = atlas[:, has_counts].copy()

Gene table: `symbol` is the current FlyBase symbol. Genes that FlyBase does not know keep their original symbol,
transgenes keep their name, and the few gene IDs without a current symbol are named by their gene ID.
`mito` marks mitochondrial genes (`mt:` symbols), `transgene` marks EGFP and gal4.

In [ ]:
gene_ids = atlas.var_names
symbol = pd.Series(gene_ids.map(gene_to_symbol), index=gene_ids, dtype="string")

is_unmapped = gene_ids.str.startswith("UNMAPPED:")
symbol[is_unmapped] = gene_ids[is_unmapped].str.removeprefix("UNMAPPED:")

is_transgene = gene_ids.isin(TRANSGENES)
symbol[is_transgene] = gene_ids[is_transgene]

symbol = symbol.fillna(pd.Series(gene_ids, index=gene_ids)).astype(str)

atlas.var["symbol"] = symbol
atlas.var["mito"] = symbol.str.startswith("mt:").to_numpy()
atlas.var["transgene"] = is_transgene

Normalisation: the raw counts are kept in `layers["counts"]`; `X` becomes log(1 + counts per 10,000).

In [ ]:
atlas.layers["counts"] = atlas.X.copy()

sc.pp.normalize_total(atlas, target_sum=10_000)
sc.pp.log1p(atlas)

In [ ]:
# text columns are stored as categories (smaller file, faster to read)
for column in atlas.obs.columns:
    atlas.obs[column] = atlas.obs[column].astype("string").astype("category")

summary = atlas.obs.groupby(["dataset", "condition"], observed=True).agg(
    cells=("sample_id", "size"),
    samples=("sample_id", "nunique"),
)
display(summary)

ATLAS_FILE.parent.mkdir(parents=True, exist_ok=True)
atlas.write_h5ad(ATLAS_FILE, compression="gzip")

print(f"saved {ATLAS_FILE.name}: {atlas.n_obs:,} cells x {atlas.n_vars:,} genes")